In [ ]:
!pip install pdfplumber

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 72.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 78.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 56.4 MB/s eta 0:00:00
  Attempting uninstall: Pillow
    Found existing installation: pillow 11.3.0
    Uninstalling pillow-11.3.0:
      Successfully uninstalled pillow-11.3.0


In [ ]:
# Data handling
import os
import re
import pandas as pd
import numpy as np

# PDF text extraction
import pdfplumber

# NLP libraries
import nltk
import spacy

# Machine Learning
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Visualization
import matplotlib.pyplot as plt

# Spelling Correction
from textblob import TextBlob

# Progress Bar
from tqdm import tqdm

In [ ]:
import nltk
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.


True

In [ ]:
!pip install spacy
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 119.4 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import spacy
nlp = spacy.load("en_core_web_sm")

In [ ]:
import zipfile
import os

zip_path = "/content/CaseStudy_Resume_Dataset.zip"
extract_path = "/content"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extracted successfully!")

Dataset extracted successfully!


In [ ]:
import os

print(os.listdir("/content"))

['.config', 'CaseStudy_Resume_Dataset.zip', 'Resume_Dataset', 'Job_Description', 'sample_data']


In [ ]:
import os

resume_folder = "/content/Resume_Dataset"

resume_files = sorted([
    os.path.join(resume_folder, file)
    for file in os.listdir(resume_folder)
    if file.lower().endswith(".pdf")
])

print("Total resumes:", len(resume_files))
print("\nFirst 5 resumes:")
for file in resume_files[:5]:
    print(file)

Total resumes: 100

First 5 resumes:
/content/Resume_Dataset/Resume_01.pdf
/content/Resume_Dataset/Resume_02.pdf
/content/Resume_Dataset/Resume_03.pdf
/content/Resume_Dataset/Resume_04.pdf
/content/Resume_Dataset/Resume_05.pdf


In [ ]:
jd_path = "/content/Job_Description/Job_Description_Data_Scientist.pdf"

print(jd_path)

/content/Job_Description/Job_Description_Data_Scientist.pdf


In [ ]:
def extract_text(pdf_path):
    text = ""

    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text()

            if page_text:
                text += page_text + "\n"

    return text

In [ ]:
resume_texts = []

for file in tqdm(resume_files):
    resume_texts.append(extract_text(file))

print("Loaded", len(resume_texts), "resumes")

100%|██████████| 100/100 [00:01<00:00, 50.17it/s]

Loaded 100 resumes


In [ ]:
jd_path = extract_text(jd_path)
print(jd_path[:500])

Job Description - Data Scientist
Location: Mumbai, India
Employment Type: Full-Time
Job Summary
Seeking a Data Scientist to analyze data, build machine learning models, and deliver
business insights using statistical and NLP techniques.
Required Skills
Python, SQL, Machine Learning, Natural Language Processing (NLP), Pandas, NumPy,
Scikit-learn, Data Visualization, Statistics, Data Cleaning
Preferred Skills
TensorFlow or PyTorch, Git, AWS/Azure/GCP, PySpark, Docker
Experience
2-4 years
Education


In [ ]:
resume_df = pd.DataFrame({
    "Resume_File": [os.path.basename(f) for f in resume_files],
    "Resume_Text": resume_texts
})

resume_df.head()

,Resume_File,Resume_Text
0,Resume_01.pdf,Candidate 1\nTarget Role: Data Scientist\nEmai...
1,Resume_02.pdf,Candidate 2\nTarget Role: Data Analyst\nEmail:...
2,Resume_03.pdf,Candidate 3\nTarget Role: ML Engineer\nEmail: ...
3,Resume_04.pdf,Candidate 4\nTarget Role: Python Developer\nEm...
4,Resume_05.pdf,Candidate 5\nTarget Role: Business Analyst\nEm...


In [ ]:
resume_df.isnull().sum()

,0
Resume_File,0
Resume_Text,0


In [ ]:
import re
import string
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [ ]:
stop_words = set(stopwords.words("english"))

lemmatizer = WordNetLemmatizer()

In [ ]:
def normalize_text(text):

    # Convert to lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)

    # Remove email addresses
    text = re.sub(r'\S+@\S+', '', text)

    # Remove phone numbers
    text = re.sub(r'\+?\d[\d\s\-()]{8,}\d', '', text)

    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))

    # Remove numbers
    text = re.sub(r'\d+', '', text)

    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    # Tokenization
    words = nltk.word_tokenize(text)

    # Remove stopwords and lemmatize
    words = [
        lemmatizer.lemmatize(word)
        for word in words
        if word not in stop_words
    ]

    return " ".join(words)

In [ ]:
import nltk

# Download all required NLTK resources
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


True

In [ ]:
resume_df["Clean_Text"] = resume_df["Resume_Text"].apply(normalize_text)

In [ ]:
resume_df[["Resume_File", "Clean_Text"]].head()

,Resume_File,Clean_Text
0,Resume_01.pdf,candidate target role data scientist email pho...
1,Resume_02.pdf,candidate target role data analyst email phone...
2,Resume_03.pdf,candidate target role ml engineer email phone ...
3,Resume_04.pdf,candidate target role python developer email p...
4,Resume_05.pdf,candidate target role business analyst email p...


In [ ]:
print("Original Resume:\n")
print(resume_df["Resume_Text"][0][:500])

print("\n\nCleaned Resume:\n")
print(resume_df["Clean_Text"][0][:500])

Original Resume:

Candidate 1
Target Role: Data Scientist
Email: candidate1@mail.com
Phone: 9810000001
Professional Summary
Results-driven professional with strong analytical, communication and technical skills.
Education
B.Tech in Computer Science (CGPA 8.1)
Experience
2 years across IT projects including analytics, automation and software development.
Projects
Customer Churn Prediction; Sales Forecasting; Resume Screening System;
Recommendation Engine.
Technical Skills
Python, SQL, Machine Learning, NLP, Pandas


Cleaned Resume:

candidate target role data scientist email phone professional summary resultsdriven professional strong analytical communication technical skill education btech computer science cgpa experience year across project including analytics automation software development project customer churn prediction sale forecasting resume screening system recommendation engine technical skill python sql machine learning nlp panda scikitlearn tensorflow git certification pyth

In [ ]:
resume_df["Clean_Text"].isnull().sum()

np.int64(0)

In [ ]:
def extract_email(text):

    pattern = r'[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}'

    match = re.search(pattern, text)

    if match:
        return match.group()

    return None

In [ ]:
def extract_phone(text):

    pattern = r'(\+?\d[\d\s\-()]{8,}\d)'

    match = re.search(pattern, text)

    if match:
        return match.group()

    return None

In [ ]:
def extract_name(text):

    lines = text.split("\n")

    for line in lines:

        line = line.strip()

        if len(line) > 2:
            return line

    return None

In [ ]:
skills_list = [

    "python",
    "sql",
    "machine learning",
    "nlp",
    "pandas",
    "scikit-learn",
    "tensorflow",
    "pytorch",
    "power bi",
    "tableau",
    "excel",
    "spark",
    "pyspark",
    "aws",
    "docker",
    "git",
    "flask",
    "fastapi"

]

In [ ]:
def extract_skills(text):

    text = text.lower()

    found = []

    for skill in skills_list:

        if skill in text:
            found.append(skill)

    return list(set(found))

In [ ]:
def extract_experience(text):

    pattern = r'(\d+)\s*\+?\s*years?'

    match = re.search(pattern, text.lower())

    if match:
        return int(match.group(1))

    return None

In [ ]:
resume_df["Name"] = resume_df["Resume_Text"].apply(extract_name)

resume_df["Email"] = resume_df["Resume_Text"].apply(extract_email)

resume_df["Phone"] = resume_df["Resume_Text"].apply(extract_phone)

resume_df["Skills"] = resume_df["Resume_Text"].apply(extract_skills)

resume_df["Experience"] = resume_df["Resume_Text"].apply(extract_experience)

In [ ]:
resume_df[[
    "Name",
    "Email",
    "Phone",
    "Experience",
    "Skills"
]].head(
)

,Name,Email,Phone,Experience,Skills
0,Candidate 1,candidate1@mail.com,9810000001,2,"[git, python, sql, pandas, scikit-learn, nlp, ..."
1,Candidate 2,candidate2@mail.com,9810000002,3,"[python, sql, tableau, power bi, excel, machin..."
2,Candidate 3,candidate3@mail.com,9810000003,4,"[aws, python, sql, docker, pytorch, tensorflow..."
3,Candidate 4,candidate4@mail.com,9810000004,5,"[git, python, sql, docker, flask, fastapi, mac..."
4,Candidate 5,candidate5@mail.com,9810000005,6,"[python, sql, tableau, power bi, excel, machin..."


In [ ]:
resume_df.to_csv("Extracted_Resume_Data.csv", index=False)

In [ ]:
sample_text = resume_df["Clean_Text"][0]

doc = nlp(sample_text)

for token in doc:
    print(token.text, "→", token.lemma_)

candidate → candidate
target → target
role → role
data → datum
scientist → scientist
email → email
phone → phone
professional → professional
summary → summary
resultsdriven → resultsdriven
professional → professional
strong → strong
analytical → analytical
communication → communication
technical → technical
skill → skill
education → education
btech → btech
computer → computer
science → science
cgpa → cgpa
experience → experience
year → year
across → across
project → project
including → include
analytics → analytic
automation → automation
software → software
development → development
project → project
customer → customer
churn → churn
prediction → prediction
sale → sale
forecasting → forecasting
resume → resume
screening → screen
system → system
recommendation → recommendation
engine → engine
technical → technical
skill → skill
python → python
sql → sql
machine → machine
learning → learn
nlp → nlp
panda → panda
scikitlearn → scikitlearn
tensorflow → tensorflow
git → git
certification → 

In [ ]:
for token in doc:
    print(token.text, token.pos_)

candidate NOUN
target NOUN
role NOUN
data NOUN
scientist NOUN
email NOUN
phone NOUN
professional ADJ
summary NOUN
resultsdriven VERB
professional ADJ
strong ADJ
analytical ADJ
communication NOUN
technical ADJ
skill NOUN
education NOUN
btech NOUN
computer NOUN
science NOUN
cgpa NOUN
experience NOUN
year NOUN
across ADP
project NOUN
including VERB
analytics NOUN
automation NOUN
software NOUN
development NOUN
project NOUN
customer NOUN
churn NOUN
prediction NOUN
sale NOUN
forecasting NOUN
resume NOUN
screening VERB
system NOUN
recommendation NOUN
engine NOUN
technical ADJ
skill NOUN
python NOUN
sql ADJ
machine NOUN
learning VERB
nlp PROPN
panda NOUN
scikitlearn VERB
tensorflow PROPN
git NOUN
certification PROPN
python PROPN
sql PROPN
cloud VERB
fundamental ADJ
machine NOUN
learning VERB
achievement NOUN
employee NOUN
recognition NOUN
hackathon PROPN
participation NOUN
process NOUN
optimization NOUN


In [ ]:
def pos_tagging(text):

    doc = nlp(text)

    tags = []

    for token in doc:
        tags.append((token.text, token.pos_))

    return tags

In [ ]:
resume_df["POS_Tags"] = resume_df["Clean_Text"].apply(pos_tagging)

In [ ]:
resume_df[["Resume_File","POS_Tags"]].head()

,Resume_File,POS_Tags
0,Resume_01.pdf,"[(candidate, NOUN), (target, NOUN), (role, NOU..."
1,Resume_02.pdf,"[(candidate, NOUN), (target, NOUN), (role, NOU..."
2,Resume_03.pdf,"[(candidate, NOUN), (target, NOUN), (role, NOU..."
3,Resume_04.pdf,"[(candidate, NOUN), (target, NOUN), (role, NOU..."
4,Resume_05.pdf,"[(candidate, NOUN), (target, NOUN), (role, NOU..."


In [ ]:
def extract_nouns(text):

    doc = nlp(text)

    nouns = []

    for token in doc:

        if token.pos_ in ["NOUN","PROPN"]:

            nouns.append(token.text)

    return nouns

In [ ]:
resume_df["Important_Words"] = resume_df["Clean_Text"].apply(extract_nouns)

In [ ]:
resume_df[[
    "Resume_File",
    "Important_Words"
]].head()

,Resume_File,Important_Words
0,Resume_01.pdf,"[candidate, target, role, data, scientist, ema..."
1,Resume_02.pdf,"[candidate, target, role, data, analyst, email..."
2,Resume_03.pdf,"[candidate, target, role, engineer, email, pho..."
3,Resume_04.pdf,"[candidate, target, role, python, developer, e..."
4,Resume_05.pdf,"[candidate, target, role, business, analyst, e..."


In [ ]:
from collections import Counter

all_tags = []

for tags in resume_df["POS_Tags"]:

    for word, tag in tags:

        all_tags.append(tag)

Counter(all_tags)

Counter({'NOUN': 3340,
         'ADJ': 420,
         'VERB': 400,
         'ADP': 50,
         'PROPN': 430,
         'AUX': 20})

In [ ]:
tfidf = TfidfVectorizer(
    stop_words='english',
    max_features=5000
)

In [ ]:
resume_vectors = tfidf.fit_transform(resume_df["Clean_Text"])

In [ ]:
print(resume_vectors.shape)

(100, 77)


In [ ]:
feature_names = tfidf.get_feature_names_out()

print(feature_names[:50])

['achievement' 'analyst' 'analytical' 'analytics' 'automation' 'aws' 'bi'
 'btech' 'business' 'candidate' 'certification' 'cgpa' 'churn' 'cloud'
 'communication' 'computer' 'customer' 'data' 'developer' 'development'
 'docker' 'education' 'email' 'employee' 'engine' 'engineer' 'excel'
 'experience' 'fastapi' 'flask' 'forecasting' 'fundamental' 'gathering'
 'git' 'hackathon' 'including' 'learning' 'machine' 'ml' 'mlops' 'nlp'
 'optimization' 'panda' 'participation' 'phone' 'postgresql' 'power'
 'prediction' 'problem' 'process']


In [ ]:
tfidf_df = pd.DataFrame(
    resume_vectors.toarray(),
    columns=feature_names
)

tfidf_df.head()

,achievement,analyst,analytical,analytics,automation,aws,bi,btech,business,candidate,...,solving,sql,statistic,strong,summary,tableau,target,technical,tensorflow,year
0,0.119880,0.000000,0.119880,0.071217,0.119880,0.000000,0.000000,0.071217,0.000000,0.071217,...,0.0,0.142435,0.000000,0.119880,0.119880,0.000000,0.071217,0.239760,0.183071,0.071217
1,0.129222,0.145977,0.129222,0.076768,0.129222,0.000000,0.197338,0.076768,0.000000,0.076768,...,0.0,0.153535,0.246978,0.129222,0.129222,0.197338,0.076768,0.258445,0.000000,0.076768
2,0.125411,0.000000,0.125411,0.074504,0.125411,0.239695,0.000000,0.074504,0.000000,0.074504,...,0.0,0.074504,0.000000,0.125411,0.125411,0.000000,0.074504,0.250823,0.191519,0.074504
3,0.125975,0.000000,0.125975,0.074838,0.125975,0.000000,0.000000,0.074838,0.000000,0.074838,...,0.0,0.074838,0.000000,0.125975,0.125975,0.000000,0.074838,0.251950,0.000000,0.074838
4,0.124423,0.140556,0.124423,0.073916,0.124423,0.000000,0.190009,0.073916,0.190009,0.073916,...,0.0,0.147833,0.000000,0.124423,0.124423,0.190009,0.073916,0.248846,0.000000,0.073916


In [ ]:
resume_index = 0

scores = zip(
    feature_names,
    resume_vectors.toarray()[resume_index]
)

top_words = sorted(
    scores,
    key=lambda x: x[1],
    reverse=True
)

print(top_words[:10])

[('learning', np.float64(0.23975997707883756)), ('machine', np.float64(0.23975997707883756)), ('professional', np.float64(0.23975997707883756)), ('technical', np.float64(0.23975997707883756)), ('nlp', np.float64(0.22912261483341986)), ('panda', np.float64(0.22912261483341986)), ('scikitlearn', np.float64(0.22912261483341986)), ('git', np.float64(0.183071469634842)), ('scientist', np.float64(0.183071469634842)), ('tensorflow', np.float64(0.183071469634842))]


In [ ]:
jd_path = "/content/Job_Description/Job_Description_Data_Scientist.pdf"

jd_text = extract_text(jd_path)
jd_clean = normalize_text(jd_text)

In [ ]:
documents = resume_df["Clean_Text"].tolist()

documents.append(jd_path_clean)

In [ ]:
tfidf = TfidfVectorizer(stop_words="english")

tfidf_matrix = tfidf.fit_transform(documents)

In [ ]:
print(tfidf_matrix.shape)

(101, 128)


In [ ]:
resume_vectors = tfidf_matrix[:-1]

job_vector = tfidf_matrix[-1]

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_scores = cosine_similarity(
    resume_vectors,
    job_vector
)

In [ ]:
resume_df["Similarity_Score"] = similarity_scores

In [ ]:
resume_df["Match_Percentage"] = (
    resume_df["Similarity_Score"] * 100
).round(2)

In [ ]:
resume_df[[
    "Resume_File",
    "Match_Percentage"
]].head()

,Resume_File,Match_Percentage
0,Resume_01.pdf,24.55
1,Resume_02.pdf,14.27
2,Resume_03.pdf,8.98
3,Resume_04.pdf,7.57
4,Resume_05.pdf,6.44


In [ ]:
ranked_candidates = resume_df.sort_values(
    by="Match_Percentage",
    ascending=False
)

ranked_candidates.reset_index(drop=True, inplace=True)

ranked_candidates["Rank"] = ranked_candidates.index + 1

In [ ]:
ranked_candidates[[
    "Rank",
    "Resume_File",
    "Match_Percentage",
    "Skills",
    "Experience"
]].head(10)

,Rank,Resume_File,Match_Percentage,Skills,Experience
0,1,Resume_01.pdf,24.55,"[git, python, sql, pandas, scikit-learn, nlp, ...",2
1,2,Resume_06.pdf,24.55,"[git, python, sql, pandas, scikit-learn, nlp, ...",7
2,3,Resume_16.pdf,24.55,"[git, python, sql, pandas, scikit-learn, nlp, ...",1
3,4,Resume_11.pdf,24.55,"[git, python, sql, pandas, scikit-learn, nlp, ...",4
4,5,Resume_21.pdf,24.55,"[git, python, sql, pandas, scikit-learn, nlp, ...",6
5,6,Resume_26.pdf,24.55,"[git, python, sql, pandas, scikit-learn, nlp, ...",3
6,7,Resume_31.pdf,24.55,"[git, python, sql, pandas, scikit-learn, nlp, ...",8
7,8,Resume_36.pdf,24.55,"[git, python, sql, pandas, scikit-learn, nlp, ...",5
8,9,Resume_41.pdf,24.55,"[git, python, sql, pandas, scikit-learn, nlp, ...",2
9,10,Resume_46.pdf,24.55,"[git, python, sql, pandas, scikit-learn, nlp, ...",7


In [ ]:
!pip install python-Levenshtein

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 34.1 MB/s eta 0:00:00


In [ ]:
from Levenshtein import distance

In [ ]:
word1 = "pythn"
word2 = "python"

distance(word1, word2)

1

In [ ]:
standard_skills = [
    "python",
    "sql",
    "machine learning",
    "nlp",
    "pandas",
    "scikit-learn",
    "tensorflow",
    "pytorch",
    "power bi",
    "tableau",
    "docker",
    "aws",
    "git"
]

In [ ]:
from rapidfuzz.distance import Levenshtein
standard_skills = [
    "python",
    "sql",
    "machine learning",
    "nlp",
    "pandas",
    "scikit-learn",
    "tensorflow",
    "pytorch",
    "power bi",
    "tableau",
    "docker",
    "aws",
    "git"
]
def correct_word(word):
   minimum_distance = float("inf")
   corrected = word
   for skill in standard_skills:

        # Compare each word inside multi-word skills
        skill_words = skill.split()

        for skill_word in skill_words:

            dist = Levenshtein.distance(
                word.lower(),
                skill_word.lower()
            )

            if dist < minimum_distance:
                minimum_distance = dist
                corrected = skill


    # allow small spelling mistakes
            if minimum_distance <= 2:
              return corrected
              return word

In [ ]:
words = [
    "pythn",
    "tensorflw",
    "machne",
    "tableu"
]


for word in words:
    print(word, "→", correct_word(word))

pythn → python
tensorflw → tensorflow
machne → machine learning
tableu → tableau


In [ ]:
def spelling_correction(text):

    words = str(text).split()

    corrected_words = []

    for word in words:

        corrected = correct_word(word)

        # If correct_word() returns None, keep the original word
        if corrected is None:
            corrected = word

        corrected_words.append(str(corrected))

    return " ".join(corrected_words)

In [ ]:
resume_df["Corrected_Text"] = (
    resume_df["Clean_Text"]
    .apply(spelling_correction))

In [ ]:
print("Before Correction:")
print(resume_df["Clean_Text"][0][:300])


print("\nAfter Correction:")
print(resume_df["Corrected_Text"][0][:300])

Before Correction:
candidate target role data scientist email phone professional summary resultsdriven professional strong analytical communication technical skill education btech computer science cgpa experience year across project including analytics automation software development project customer churn prediction 

After Correction:
candidate target role data scientist email phone professional summary resultsdriven professional strong analytical communication technical skill education btech computer science cgpa experience year across project including analytics automation software development project customer churn prediction 


In [ ]:
documents_corrected = (
    resume_df["Corrected_Text"]
    .tolist()
)

documents_corrected.append(
    jd_path_clean
)

In [ ]:
tfidf_corrected = TfidfVectorizer(
    stop_words="english"
)


tfidf_matrix_corrected = (
    tfidf_corrected
    .fit_transform(documents_corrected)
)

In [ ]:
resume_vectors_corrected = (
    tfidf_matrix_corrected[:-1]
)


job_vector_corrected = (
    tfidf_matrix_corrected[-1]
)


new_scores = cosine_similarity(
    resume_vectors_corrected,
    job_vector_corrected
)

In [ ]:
resume_df["Updated_Match_Score"] = (
    new_scores * 100
).round(2)

In [ ]:
resume_df[[
    "Resume_File",
    "Match_Percentage",
    "Updated_Match_Score"
]].head()

,Resume_File,Match_Percentage,Updated_Match_Score
0,Resume_01.pdf,24.55,21.43
1,Resume_02.pdf,14.27,15.48
2,Resume_03.pdf,8.98,11.38
3,Resume_04.pdf,7.57,8.85
4,Resume_05.pdf,6.44,8.40


In [ ]:
resume_df["Final_Score"] = (
    resume_df["Updated_Match_Score"]
).round(2)

In [ ]:
ranked_candidates = (
    resume_df
    .sort_values(
        by="Final_Score",
        ascending=False
    )
    .reset_index(drop=True)
)


ranked_candidates["Rank"] = (
    ranked_candidates.index + 1
)


ranked_candidates[
    [
        "Rank",
        "Resume_File",
        "Final_Score"
    ]
].head(10)

,Rank,Resume_File,Final_Score
0,1,Resume_01.pdf,21.43
1,2,Resume_06.pdf,21.43
2,3,Resume_16.pdf,21.43
3,4,Resume_11.pdf,21.43
4,5,Resume_21.pdf,21.43
5,6,Resume_26.pdf,21.43
6,7,Resume_31.pdf,21.43
7,8,Resume_36.pdf,21.43
8,9,Resume_41.pdf,21.43
9,10,Resume_46.pdf,21.43


In [ ]:
def assign_category(score):
    if score >= 90:
        return "Excellent Match"
    elif score >= 75:
        return "Good Match"
    elif score >= 60:
        return "Average Match"
    else:
        return "Low Match"

ranked_candidates["Category"] = ranked_candidates["Final_Score"].apply(assign_category)

In [ ]:
resume_df[
    [
        "Resume_File",
        "Final_Score",
        "Category"
    ]
].head(10)

,Resume_File,Final_Score,Category
0,Resume_01.pdf,21.43,High Match
1,Resume_02.pdf,15.48,High Match
2,Resume_03.pdf,11.38,Medium Match
3,Resume_04.pdf,8.85,Low Match
4,Resume_05.pdf,8.40,Low Match
5,Resume_051.pdf,17.86,High Match
6,Resume_052.pdf,14.31,Medium Match
7,Resume_053.pdf,4.17,Low Match
8,Resume_054.pdf,4.25,Low Match
9,Resume_055.pdf,5.99,Low Match


In [ ]:
job_skills = [
    "python",
    "sql",
    "machine learning",
    "nlp",
    "pandas",
    "scikit-learn",
    "tensorflow",
    "tableau"
]

def skill_score(text):
    text = text.lower()
    matched = sum(skill in text for skill in job_skills)
    return (matched / len(job_skills)) * 100

resume_df["Skill_Score"] = resume_df["Corrected_Text"].apply(skill_score)

In [ ]:
def experience_score(exp):
    try:
        exp = float(exp)
    except:
        return 0

    if exp >= 4:
        return 100
    elif exp >= 2:
        return 80
    elif exp >= 1:
        return 60
    else:
        return 40

resume_df["Experience_Score"] = resume_df["Experience"].apply(experience_score)

In [ ]:
def education_score(text):
    text = text.lower()

    if "master" in text or "m.tech" in text:
        return 100
    elif "b.tech" in text or "bachelor" in text:
        return 80
    else:
        return 50

resume_df["Education_Score"] = resume_df["Resume_Text"].apply(education_score)

In [ ]:
def certification_score(text):
    text = text.lower()

    certifications = [
        "aws",
        "google",
        "microsoft",
        "python",
        "tensorflow"
    ]

    matched = sum(cert in text for cert in certifications)

    return (matched / len(certifications)) * 100

resume_df["Certification_Score"] = resume_df["Resume_Text"].apply(certification_score)

In [ ]:
resume_df["Final_Score"] = (
    0.50 * resume_df["Skill_Score"] +
    0.25 * resume_df["Experience_Score"] +
    0.15 * resume_df["Education_Score"] +
    0.10 * resume_df["Certification_Score"]
).round(2)

In [ ]:
ranked_candidates = (
    resume_df
    .sort_values(by="Final_Score", ascending=False)
    .reset_index(drop=True)
)

ranked_candidates["Rank"] = ranked_candidates.index + 1

In [ ]:
def assign_category(score):
    if score >= 85:
        return "Excellent Match"
    elif score >= 70:
        return "Strong Match"
    elif score >= 55:
        return "Good Match"
    elif score >= 40:
        return "Average Match"
    else:
        return "Low Match"

ranked_candidates["Category"] = ranked_candidates["Final_Score"].apply(assign_category)

In [ ]:
top_candidates = ranked_candidates.head(10)

top_candidates[
    [
        "Rank",
        "Resume_File",
        "Skill_Score",
        "Experience_Score",
        "Education_Score",
        "Certification_Score",
        "Final_Score",
        "Category"
    ]
]

,Rank,Resume_File,Skill_Score,Experience_Score,Education_Score,Certification_Score,Final_Score,Category
0,1,Resume_06.pdf,87.5,100,80,40.0,84.75,Strong Match
1,2,Resume_31.pdf,87.5,100,80,40.0,84.75,Strong Match
2,3,Resume_46.pdf,87.5,100,80,40.0,84.75,Strong Match
3,4,Resume_11.pdf,87.5,100,80,40.0,84.75,Strong Match
4,5,Resume_21.pdf,87.5,100,80,40.0,84.75,Strong Match
5,6,Resume_36.pdf,87.5,100,80,40.0,84.75,Strong Match
6,7,Resume_26.pdf,87.5,80,80,40.0,79.75,Strong Match
7,8,Resume_41.pdf,87.5,80,80,40.0,79.75,Strong Match
8,9,Resume_01.pdf,87.5,80,80,40.0,79.75,Strong Match
9,10,Resume_16.pdf,87.5,60,80,40.0,74.75,Strong Match
